In [7]:
import csv
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options


BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# Chrome configuration
options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

books = []

try:
    for page in range(1, 6):
        url = BASE_URL.format(page)
        print(f"Scraping page {page}: {url}")

        driver.get(url)
        time.sleep(1)

        book_elements = driver.find_elements(
            By.CSS_SELECTOR,
            "article.product_pod"
        )

        for book in book_elements:

            # Title
            title_element = book.find_element(By.CSS_SELECTOR, "h3 a")
            title = title_element.get_attribute("title")

            # URL
            book_url = title_element.get_attribute("href")

            # Price
            price_text = book.find_element(
                By.CSS_SELECTOR,
                ".price_color"
            ).text

            price = float(
                price_text.replace("£", "").strip()
            )

            # Rating
            rating_element = book.find_element(
                By.CSS_SELECTOR,
                "p.star-rating"
            )

            rating_classes = rating_element.get_attribute("class").split()

            rating_word = next(
                x for x in rating_classes
                if x != "star-rating"
            )

            rating = rating_map[rating_word]

            # Stock
            stock_text = book.find_element(
                By.CSS_SELECTOR,
                ".availability"
            ).text.strip()

            in_stock = "In stock" in stock_text

            books.append({
                "title": title,
                "price": price,
                "rating": rating,
                "in_stock": in_stock,
                "url": book_url
            })

finally:
    driver.quit()


# Make sure exactly 100 books were collected
assert len(books) == 100, f"Expected 100 books, got {len(books)}"


# Write CSV
with open(
    r"C:\Users\mostafa.hussin\Desktop\Books.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "title",
            "price",
            "rating",
            "in_stock",
            "url"
        ]
    )

    writer.writeheader()
    writer.writerows(books)


print(f"Done. {len(books)} books saved to books.csv")


Scraping page 1: https://books.toscrape.com/catalogue/page-1.html
Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
Scraping page 5: https://books.toscrape.com/catalogue/page-5.html
Done. 100 books saved to books.csv
